May need more spin up. Also need to discard spin up period. See if you want daily values for some too.

In [1]:
import numpy as np
import os
import xarray as xr
import Ngl
from matplotlib.dates import DateFormatter, HourLocator, date2num, num2date

DIR = '/glade/derecho/scratch/nforcone/CAM_6_4_025_20240829_ne16_ne16_mg17_Aqua_Perturbation/run'
FILE = 'CAM_6_4_025_20240829_ne16_ne16_mg17_Aqua_Perturbation.cam.h0a.0001-01-31-00000.nc'
ds = xr.open_dataset(os.path.join(DIR, FILE), decode_cf=False)
ds

<xarray.Dataset> Size: 38MB
Dimensions:       (lat: 91, lon: 180, lev: 30, ilev: 31, trop_pref: 30,
                   trop_prefi: 31, time: 1, nbnd: 2, chars: 8)
Coordinates:
  * lat           (lat) float64 728B -90.0 -88.0 -86.0 -84.0 ... 86.0 88.0 90.0
  * lon           (lon) float64 1kB 0.0 2.0 4.0 6.0 ... 352.0 354.0 356.0 358.0
  * lev           (lev) float64 240B 3.643 7.595 14.36 ... 957.5 976.3 992.6
  * ilev          (ilev) float64 248B 2.255 5.032 10.16 ... 967.5 985.1 1e+03
  * trop_pref     (trop_pref) float64 240B 3.643 7.595 14.36 ... 976.3 992.6
  * trop_prefi    (trop_prefi) float64 248B 2.255 5.032 10.16 ... 985.1 1e+03
  * time          (time) float64 8B 15.0
Dimensions without coordinates: nbnd, chars
Data variables: (12/44)
    w             (lat) float64 728B ...
    hyam          (lev) float64 240B ...
    hybm          (lev) float64 240B ...
    hyai          (ilev) float64 248B ...
    hybi          (ilev) float64 248B ...
    date          (time) int32 4B ...
    ...            ...
    VV            (time, lev, lat, lon) float32 2MB ...
    Z3            (time, lev, lat, lon) float32 2MB ...
    ZZ            (time, lev, lat, lon) float32 2MB ...
    TT            (time, lev, lat, lon) float32 2MB ...
    VQ            (time, lev, lat, lon) float32 2MB ...
    QQ            (time, lev, lat, lon) float32 2MB ...
Attributes:
    interp_type:       bilinear
    interp_outputgri:  equally spaced with poles
    Conventions:       CF-1.0
    source:            CAM
    case:              CAM_6_4_025_20240829_ne16_ne16_mg17_Aqua_Perturbation
    logname:           nforcone
    host:              dec2411
    initial_file:      /glade/campaign/cesm/cesmdata/inputdata/atm/cam/inic/s...
    topography_file:   bnd_topo
    model_doi_url:     not_set
    time_period_freq:  day_30

$$\overline{vT}=\overline{v}\overline{T}+\overline{v'T'}$$

overbar is time mean, ' is time anomaly

$$[vT]=[v][T]+[v^*T^*]$$

[ ] is zonal mean, * is zonal anomaly

In [13]:
# extract variables (need numpy arrays for vertical interpolation)
hyam = ds["hyam"].to_numpy()
hybm = ds["hybm"].to_numpy()
V    = ds["V"].to_numpy()   # [time: 240, lev: 30, lat: 257, lon: 512]
T    = ds["T"].to_numpy()   # [time: 240, lev: 30, lat: 257, lon: 512]
VT   = ds["VT"].to_numpy()  # [time: 240, lev: 30, lat: 257, lon: 512]
psrf = ds["PS"].to_numpy()  # [time: 240, lat: 257, lon: 512] MUST BE Pa
p0   = 1000                 # MUST BE hPa
lats = ds["lat"].to_numpy()
lons = ds["lon"].to_numpy()

# define the output pressure levels.
pnew = np.array([976, 860, 763, 525, 233])  # MUST BE hPa

# interpolation settings
intyp = 1      # 1=linear, 2=log, 3=log-log
kxtrp = False  # True=extrapolate (when the output pressure level is outside of the range of psrf)

# vertical interpolation
V_interp = Ngl.vinth2p(V, hyam, hybm, pnew, psrf, intyp, p0, 1, kxtrp)
V_interp[V_interp==1e30] = np.nan

T_interp = Ngl.vinth2p(T, hyam, hybm, pnew, psrf, intyp, p0, 1, kxtrp)
T_interp[T_interp==1e30] = np.nan

VT_interp = Ngl.vinth2p(VT, hyam, hybm, pnew, psrf, intyp, p0, 1, kxtrp)
VT_interp[VT_interp==1e30] = np.nan

time_dt = num2date(ds["time"])

# create new xarray DataArray
coords_dict = {'time': time_dt,
               'plev': pnew,
               'lat': lats,
               'lon': lons}

dim_names = ['time', 'plev', 'lat', 'lon']
V_interp_da = xr.DataArray(V_interp, dims=dim_names, coords=coords_dict)
T_interp_da = xr.DataArray(T_interp, dims=dim_names, coords=coords_dict)
VT_interp_da = xr.DataArray(VT_interp, dims=dim_names, coords=coords_dict)

In [15]:
V_zonal_mean = V_interp_da.mean(dim='lat')